<a href="https://colab.research.google.com/github/pritika-v/ZYLiQ_AI_Internship/blob/main/Data_Extraction_from_wordFiles(9_6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re #to find rtf patterns
from pprint import pprint #to print nested lists neatly

In [2]:
file_path="/content/3_ZYL_Table 14.3.2.1.1_AE_Death 3.rtf"

# Open file in read mode
with open( file_path,"r",encoding="utf-8", errors="ignore") as f:
    rtf_text = f.read() # Read entire RTF file into memory

Accessing the table rows directly

In [3]:
#subject ids are usually 8 digits
subject_match=re.search(r"\b\d{8}\b",rtf_text)
if not subject_match:
  raise ValueError("No Subject ID found")

In [4]:
#get the position where the subject id starts
start_pos=subject_match.start()

#keep only that data section and ignore the headers above taht point
data_section=rtf_text[start_pos:]

In [5]:
# in rtf, rows end with \row. therefore split rows on \row.
raw_rows=data_section.split(r"\row")


In [6]:
''' List comprehension
new_list = []

for item in iterable:
    if condition:
        new_list.append(expression)'''
#Remove whitespaces and empty rows
raw_rows=[
     row.strip() #removes white spaces
     for row in raw_rows
     if row.strip()
 ]

EXTRACTING CELLS FROM EACH ROW

In [7]:
all_rows=[]

for row in raw_rows:
  cells=row.split(r"\cell") #splitting row into cells
  cleaned_cells=[]
  for cell in cells:
    cell=re.sub( r"\\[a-z]+\d* ?"," ",cell) #removing the RTF format
    cell=cell.replace(r"\line","\n") #Convert RTF line breaks into newlines
    cell=cell.replace("{","") # REmoving braces
    cell=cell.replace("}","")
    cell=re.sub(r"\s+"," ", cell) # Collapse multiple spaces
    cell=cell.strip() #Removing keading and trailing spaces
    if re.search(r"Width\d+", cell):
        continue
    if re.search(r"x\d+", cell):
        continue
    #Clearing the metadata
    if cell.startswith("\\* 504b03"):
      continue
    if "Root Entry" in cell:
      continue
    if "xml version" in cell:
      continue
    cleaned_cells.append(cell) #keep only non-empty cells

  #Convert all cels into one text
  row_text=" ".join(cleaned_cells)

  #check for junk rows (metadata)
  junk_patterns=["width","cellx","504b3","Root Entry","theme","xml version"]
  is_junk=False
  for pattern in junk_patterns:
    if pattern.lower() in row_text.lower():
      is_junk=True
      break
  if is_junk: #Skip the junk rows
    continue

  #Only rows that start with subject id or Treatment
  if (
        re.search(r"\b\d{8}\b", row_text)
        or "Treatment/Treatment" in row_text
    ):
        all_rows.append(cleaned_cells)




In [8]:
print("="*100)
print("Total Rows Found")
print("="*100)

print(len(all_rows))
print()

print("="*100)
print("All rows")
print("="*100)

for i, row in enumerate(all_rows):
  print(f"\nROW{i+1}")
  print("-"*80)
  pprint(row) #will add indetntaion, nested structures easier to read

rows=all_rows


Total Rows Found
1

All rows

ROW1
--------------------------------------------------------------------------------
['10011028 28, F , B',
 'Treatment/Treatment',
 'Investigations/ Virus identification and serology/ SARS-CoV-2 test positive/ '
 'Covid-19 Positive',
 '2021-04-08 20:00 / 28 - 2021-04- 08 20:00 / 28',
 '',
 'Yes',
 '',
 'Severe/ Not Related',
 'Not Applicable/ -',
 'Fatal']


Extracting the header columns now

In [9]:
header_start=rtf_text.find("Subject ID")
print(header_start)

#Find the first row ending \row after header start
header_end= rtf_text.find(r"\row", header_start)

header_block=rtf_text[header_start: header_end]
print(header_block)

29590
Subject ID\line Age(y),\line Sex, Race
\cell \hich\af5\dbch\af31505\loch\f5 Study\line Phase/\line Period of\line AE Onset\cell \hich\af5\dbch\af31505\loch\f5 System Organ Class/\line High Level Term/\line Preferred Term/\line Event\cell \hich\af5\dbch\af31505\loch\f5 AE Start Date\line 
Time/Study Day\line -\line AE End Date\line Time/Study Day\cell \hich\af5\dbch\af31505\loch\f5 AE\line Duration\line (days)}{\rtlch\fcs1 \af5\afs16 \ltrch\fcs0 \f5\fs16\cf1\super\insrsid7406573 \hich\af5\dbch\af31505\loch\f5 a}{\rtlch\fcs1 \af5\afs16 
\ltrch\fcs0 \f5\fs16\cf1\insrsid7406573 \cell \hich\af5\dbch\af31505\loch\f5 Serious\line Criteria\cell \hich\af5\dbch\af31505\loch\f5 Associated with Overdose?\cell \hich\af5\dbch\af31505\loch\f5 Severity/\line Relation\line to Study\line drug\cell 
\hich\af5\dbch\af31505\loch\f5 Study Drug Action\line Taken/Other\line \hich\af5\dbch\af31505\loch\f5 Action Taken\cell \hich\af5\dbch\af31505\loch\f5 Outcome\cell }\pard \ltrpar\ql \li0\ri0\sa160\sl278

In [10]:
# Split into cells
header_cells = header_block.split(r"\cell")

raw_headers = []

for cell in header_cells:

    # Remove RTF commands
    cell = re.sub(
        r"\\[a-z]+\d* ?",
        " ",
        cell
    )

    # Convert line breaks
    cell = cell.replace(r"\line", "\n")

    # Remove braces
    cell = cell.replace("{", "")
    cell = cell.replace("}", "")

    #Remove trailing spaces from each line
    lines=[
        x.strip()
        for x in cell.split("\n")
    ]

    #remove empty lines
    lines=[
        x
        for x in lines
        if x
    ]

    #Rebuild while preserving line breaks
    cell="\n".join(lines)

    #skip empty cells
    if not cell:
      continue

    cell = cell.strip()

    if cell:
        raw_headers.append(cell)

    #removing metadat cells
    filtered_headers=[]
    for header in raw_headers:
      if "Width" in header:
        continue #skip that cell with width metadata
      if re.search(r"x\d+",header):
        continue #skip position metadata
      filtered_headers.append(header)
    raw_headers=filtered_headers

print("="*80)
print("RAW HEADER STRUCTURE")
print("="*80)

for i, h in enumerate(raw_headers):

    print("\n")
    print(f"HEADER {i}")
    print("-"*50)

    print(h)

RAW HEADER STRUCTURE


HEADER 0
--------------------------------------------------
Subject ID Age(y), Sex, Race


HEADER 1
--------------------------------------------------
Study Phase/ Period of AE Onset


HEADER 2
--------------------------------------------------
System Organ Class/ High Level Term/ Preferred Term/ Event


HEADER 3
--------------------------------------------------
AE Start Date
Time/Study Day - AE End Date Time/Study Day


HEADER 4
--------------------------------------------------
AE Duration (days)                 a


HEADER 5
--------------------------------------------------
Serious Criteria


HEADER 6
--------------------------------------------------
Associated with Overdose?


HEADER 7
--------------------------------------------------
Severity/ Relation to Study drug


HEADER 8
--------------------------------------------------
Study Drug Action Taken/Other       Action Taken


HEADER 9
--------------------------------------------------
Outcome


Building Schema

In [11]:
import re
def build_schema(headers):
  schema=[]
  for header in headers:
    header=re.sub(r"\s+"," ", header).strip()
    if "Subject ID" in header and "Age" in header:
      schema.append(["Subject ID","Age(y)","Sex","Race"])
      continue
    if (
    "AE Start Date" in header
    and "AE End Date" in header):

      schema.append([
          "AE Start Date Time",
          "AE Start Study Day",
          "AE End Date Time",
          "AE End Study Day"
      ])

      continue
    #split the "/" in header"
    if "/" in header:
      parts=[x.strip()
      for x in header.split("/")] #result--['AE Start Date', 'Time', 'Study Day']
      parts=[p for p in parts if p] #filtering empty strings
      schema.append(parts)
      continue
    schema.append([header])
  return schema

In [12]:
schema=build_schema(raw_headers)
from pprint import pprint
pprint(schema)

[['Subject ID', 'Age(y)', 'Sex', 'Race'],
 ['Study Phase', 'Period of AE Onset'],
 ['System Organ Class', 'High Level Term', 'Preferred Term', 'Event'],
 ['AE Start Date Time',
  'AE Start Study Day',
  'AE End Date Time',
  'AE End Study Day'],
 ['AE Duration (days) a'],
 ['Serious Criteria'],
 ['Associated with Overdose?'],
 ['Severity', 'Relation to Study drug'],
 ['Study Drug Action Taken', 'Other Action Taken'],
 ['Outcome']]


Mapping Headers and Rows

In [13]:
for i, row in enumerate(all_rows):

    print("\nROW", i+1)
    print("Number of cells =", len(row))

    for j, cell in enumerate(row):
        print(j, "->", repr(cell))


ROW 1
Number of cells = 10
0 -> '10011028 28, F , B'
1 -> 'Treatment/Treatment'
2 -> 'Investigations/ Virus identification and serology/ SARS-CoV-2 test positive/ Covid-19 Positive'
3 -> '2021-04-08 20:00 / 28 - 2021-04- 08 20:00 / 28'
4 -> ''
5 -> 'Yes'
6 -> ''
7 -> 'Severe/ Not Related'
8 -> 'Not Applicable/ -'
9 -> 'Fatal'


In [14]:
records=[]
last_subject_info=None #Stores subject info for carryforward rows

In [15]:
import re
import json

records = []

last_subject_info = None#for previous subject info

for row in all_rows:

    record = {}

    for header_group, cell_value in zip(schema, row):

        if header_group == ["Subject ID","Age(y)","Sex","Race"]:

            if cell_value.strip():

                match = re.match(
                    r"(\d+)\s+(\d+),\s*([^,]+),\s*(.+)",
                    cell_value
                )

                if match:

                    values = [
                        match.group(1), #groups of text gotten from re
                        match.group(2),
                        match.group(3).strip(),
                        match.group(4).strip()
                    ]

                    for key, value in zip(header_group,values):
                        record[key] = value

            continue

        # =============================================
        # DATE COLUMN
        # =============================================

        if header_group == [
            "AE Start Date Time",
            "AE Start Study Day",
            "AE End Date Time",
            "AE End Study Day"
        ]:

            match = re.match(
                r"(.*?)\s*/\s*(\d+)\s*-\s*(.*?)\s*/\s*(\d+)",
                cell_value
            )

            if match:

                values = [
                    match.group(1).strip(),
                    match.group(2).strip(),
                    match.group(3).strip(),
                    match.group(4).strip()
                ]

                for key, value in zip(header_group,values):
                    record[key] = value

            continue

        #single key header
        if len(header_group) == 1:
            record[header_group[0]] = cell_value.strip()
            continue
        values = [
            x.strip()
            for x in cell_value.split("/")
        ]

        # remove empty strings
        values = [
            x
            for x in values
            if x
        ]

        # pad missing values
        while len(values) < len(header_group):

            values.append("")

        # map dynamically
        for key, value in zip(
            header_group,
            values
        ):

            record[key] = value

    if record.get("Subject ID"):

        last_subject_info = {

            "Subject ID":
                record.get("Subject ID"),

            "Age(y)":
                record.get("Age(y)"),

            "Sex":
                record.get("Sex"),

            "Race":
                record.get("Race")
        }

    else:

        if last_subject_info:

            record.update(last_subject_info)
    records.append(record)

print(
    json.dumps(
        records,
        indent=4,
        ensure_ascii=False
    )
)

[
    {
        "Subject ID": "10011028",
        "Age(y)": "28",
        "Sex": "F",
        "Race": "B",
        "Study Phase": "Treatment",
        "Period of AE Onset": "Treatment",
        "System Organ Class": "Investigations",
        "High Level Term": "Virus identification and serology",
        "Preferred Term": "SARS-CoV-2 test positive",
        "Event": "Covid-19 Positive",
        "AE Start Date Time": "2021-04-08 20:00",
        "AE Start Study Day": "28",
        "AE End Date Time": "2021-04- 08 20:00",
        "AE End Study Day": "28",
        "AE Duration (days) a": "",
        "Serious Criteria": "Yes",
        "Associated with Overdose?": "",
        "Severity": "Severe",
        "Relation to Study drug": "Not Related",
        "Study Drug Action Taken": "Not Applicable",
        "Other Action Taken": "-",
        "Outcome": "Fatal"
    }
]
